# nn.Module

`super().__init__()` is required to call the `__init__` of the parent class (`nn.Module`). It creates the internal dicts (`_parameters`, `_modules`, `_buffers`) that track everything you register.

Without it, the weights are not stored in the default places (`_parameters`, `_modules`, `_buffers`), because those dicts are never created. Then `__setattr__` has nowhere to register modules and parameters, so `parameters()`, `state_dict()`, `.to(device)` and the optimizer cannot find them.

- Without it: `AttributeError: cannot assign module before Module.__init__() call`.
- Call the model with `model(x)`, not `model.forward(x)`, so hooks run.

In [1]:
import torch.nn as nn

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        

# Args

here we use the `nn.X()` to create **modules** inside `__init__`.

### 1. Modules: `nn.X()`

Every layer (`nn.Linear`, `nn.Conv2d`, `nn.Embedding`, ...) is itself a subclass of `nn.Module`. When we write `nn.Linear(4, 2)`, the layer creates and initializes its own learnable tensors (`weight` and `bias`) internally as `nn.Parameter`. We never build them by hand.

Modules can be nested: `Net` contains `fc`, and `fc` contains `weight` and `bias`. This forms a tree, and methods like `parameters()`, `state_dict()` and `.to(device)` walk this tree recursively.

### 2. How registration works

`nn.Module` overrides `__setattr__`. When we do `self.something = value`, it checks the type of `value`:

- `nn.Module` -> stored in `self._modules["something"]`
- `nn.Parameter` -> stored in `self._parameters["something"]`
- anything else -> stored as a normal Python attribute

This is why `super().__init__()` is required: it creates `_modules`, `_parameters` and `_buffers`. Without them, `__setattr__` has nowhere to register things.

Only registered objects are seen by `parameters()`, `state_dict()`, `.to()`, `.train()/.eval()` and the optimizer.

### 3. Plain tensors vs `nn.Parameter`

A plain tensor is a valid attribute, but it is invisible to the module machinery.

```python
self.t = torch.randn(2, 3)
```

- not in `parameters()`, so the optimizer never updates it
- not in `state_dict()`, so it is not saved or loaded
- not moved by `.to(device)` / `.cuda()`, which causes device mismatch errors

To make a tensor **learnable**, wrap it in `nn.Parameter`:

```python
self.w = nn.Parameter(torch.randn(2, 3))
```

`nn.Parameter` is a `torch.Tensor` subclass with `requires_grad=True` by default, and it tells the module "register this as a parameter".

### 4. Tensors that are state but not learnable: buffers

Some tensors must be saved and moved with the model but must **not** be trained (e.g. running mean/var in BatchNorm, positional encodings, masks). For these we use `register_buffer`:

- in `buffers()` and `state_dict()`
- moved by `.to(device)`
- not in `parameters()`, so the optimizer ignores it

### 5. Summary

| Assigned in `__init__` | Registered | Stored in | Trained | In `state_dict` | Moved by `.to()` |
|---|---|---|---|---|---|
| `nn.X()` (any module) | yes | `_modules` | via its params | yes (its params/buffers) | yes |
| `nn.Parameter(t)` | yes | `_parameters` | yes | yes | yes |
| `register_buffer(name, t)` | yes | `_buffers` | no | yes | yes |
| plain `torch.Tensor` | no | normal attr | no | no | no |
| module inside a Python `list` | no | normal attr | no | no | no |

### 6. Containers

A Python `list` or `dict` of modules is not registered, because `__setattr__` only sees the list object, not what is inside. Use the containers that are themselves modules:

- `nn.ModuleList`, `nn.ModuleDict`, `nn.Sequential` for modules
- `nn.ParameterList`, `nn.ParameterDict` for parameters


In [46]:
import torch.nn as nn
import torch

class Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.linear = nn.Linear(10, 10)
        self.embedding = nn.Embedding(10, 10)
        self.blocks = nn.ModuleList([nn.Linear(2, 2) for _ in range(3)])
        self.parameter = nn.Parameter(torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0]))


model = Model()

# Params

.parameters()
.named_parameters()

Both iterate over the learnable tensors (`nn.Parameter`) registered in the module, **recursively** through all submodules.

### `.parameters()`

Returns a **generator** of `nn.Parameter`, with no names.

- Typical use: `torch.optim.Adam(model.parameters(), lr=1e-3)`.
- It is a generator, so it is **consumed after one pass**. Use `list(model.parameters())` to reuse it.
- An empty loop means nothing was registered (e.g. layers in a plain Python list instead of `nn.ModuleList`) or the generator was already consumed.

### `.named_parameters()`

Returns a generator of `(name, parameter)` tuples.

- Names are built on the fly from the keys of `_modules` and `_parameters`, joined by dots. They are not stored anywhere.
- The name comes from the **attribute name** used in `__init__`. Renaming `self.fc` to `self.linear` changes `fc.weight` to `linear.weight` and can break old checkpoints.
- The parameter itself does not know its name. To find it, compare by identity: `next(n for n, p in model.named_parameters() if p is target)`.


To read or copy values: `p.detach()`, `p.detach().cpu().numpy()`, `p.detach().clone()`.

## Common patterns

```python
# count parameters
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

# freeze everything except the layers whose name starts with "fc"
for name, p in model.named_parameters():
    p.requires_grad = name.startswith("fc")

# access one by name
model.get_parameter("block.0.weight")
```

## Notes

- Includes parameters with `requires_grad=False`.
- Shared parameters (weight tying) are returned **only once** (first path found). `state_dict()` lists every name.
- Buffers (`register_buffer`) are not included. See `named_buffers()` or `state_dict()`.

In [36]:
params = model.parameters()
named_params = model.named_parameters()


print(modules)
print(params)
print(named_params)

<generator object Module.modules at 0x000001CC4DB676A0>
<generator object Module.parameters at 0x000001CC4DB67010>
<generator object Module.named_parameters at 0x000001CC4DBED470>


In [37]:
for param in params:
    print(param)
    print(param.shape)
    print('==================')

Parameter containing:
tensor([1., 2., 3., 4., 5.], requires_grad=True)
torch.Size([5])
Parameter containing:
tensor([[-0.0760,  0.2177, -0.2792, -0.2875,  0.2549, -0.0866,  0.1871,  0.0258,
         -0.1009, -0.2125],
        [-0.2706, -0.3055,  0.0255,  0.1647,  0.2868,  0.1893,  0.2385, -0.0838,
         -0.1555, -0.0729],
        [-0.1224, -0.2392,  0.2829, -0.1563, -0.2887, -0.1084,  0.1086,  0.1388,
          0.0758,  0.0004],
        [ 0.1101,  0.1111,  0.1270, -0.0375,  0.1240, -0.2399, -0.1925, -0.2981,
          0.1103, -0.0683],
        [ 0.1829,  0.2501, -0.1380,  0.0014, -0.0603, -0.1334, -0.1009,  0.0230,
         -0.2332,  0.0056],
        [ 0.1474, -0.2395,  0.2186,  0.0255, -0.2976, -0.0423,  0.0987, -0.2166,
         -0.1479,  0.2277],
        [-0.0356,  0.1711, -0.1023,  0.1030,  0.0557,  0.0039, -0.1533,  0.1365,
         -0.0266, -0.1698],
        [-0.2443, -0.0755,  0.2951,  0.1592, -0.2272, -0.1218, -0.0562,  0.2821,
         -0.2913,  0.1915],
        [ 0.2597, -

In [38]:
for name, param in named_params:
    print(name)
    print(param)
    print('=========')

parameter
Parameter containing:
tensor([1., 2., 3., 4., 5.], requires_grad=True)
linear.weight
Parameter containing:
tensor([[-0.0760,  0.2177, -0.2792, -0.2875,  0.2549, -0.0866,  0.1871,  0.0258,
         -0.1009, -0.2125],
        [-0.2706, -0.3055,  0.0255,  0.1647,  0.2868,  0.1893,  0.2385, -0.0838,
         -0.1555, -0.0729],
        [-0.1224, -0.2392,  0.2829, -0.1563, -0.2887, -0.1084,  0.1086,  0.1388,
          0.0758,  0.0004],
        [ 0.1101,  0.1111,  0.1270, -0.0375,  0.1240, -0.2399, -0.1925, -0.2981,
          0.1103, -0.0683],
        [ 0.1829,  0.2501, -0.1380,  0.0014, -0.0603, -0.1334, -0.1009,  0.0230,
         -0.2332,  0.0056],
        [ 0.1474, -0.2395,  0.2186,  0.0255, -0.2976, -0.0423,  0.0987, -0.2166,
         -0.1479,  0.2277],
        [-0.0356,  0.1711, -0.1023,  0.1030,  0.0557,  0.0039, -0.1533,  0.1365,
         -0.0266, -0.1698],
        [-0.2443, -0.0755,  0.2951,  0.1592, -0.2272, -0.1218, -0.0562,  0.2821,
         -0.2913,  0.1915],
        [ 0

# Curiosity

Some layers store their weights in a different shape from the one we use to create them. The classic example is `nn.Linear`:

- We create it as `nn.Linear(in, out)`, but the weight is stored as `(out, in)`.
- On the forward pass (`y = layer(x)`), the weight is transposed on the fly: `y = x @ W.T + b`. The stored tensor never changes, only the computation uses the transpose.

```python
layer = nn.Linear(4, 2)     # in=4, out=2
layer.weight.shape          # torch.Size([2, 4])  -> (out, in)
```

> Weight tying works out of the box because of this convention.
> - `nn.Embedding(vocab, dmodel)` creates and stores the weight as `(vocab, dmodel)`.
> - The LM head is created as `nn.Linear(dmodel, vocab)` but stores its weight as `(vocab, dmodel)`.
> - Both shapes are identical, so we can share the same tensor with no transpose:


- `named_parameters()` returns the shared tensor only once (first path found), so the optimizer trains it a single time.
- `state_dict()` lists both names, pointing to the same tensor.

In [45]:
class WeightTying(nn.Module):
    def __init__(self, vocab_size = 2, dmodel = 3):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, dmodel)
        self.linear = self.embedding.weight

m  = WeightTying(2, 3)

for name, param in m.named_parameters():
    print(param.shape)

torch.Size([2, 3])


# Forward Method

### What `forward` is

`forward` is the method where you **define the computation** of an `nn.Module`: how inputs become outputs. It is the one method you are expected to override when you subclass `nn.Module`.


### `__call__` is what runs it

When you write `model(x)`, Python invokes `model.__call__(x)`. In `nn.Module`, that is implemented as `_call_impl`, which wraps `forward`:

```python
def __call__(self, *args, **kwargs):
    # 1. run forward pre-hooks (can modify inputs)
    result = self.forward(*args, **kwargs)
    # 2. run forward hooks (can inspect/modify outputs)
    # 3. attach backward hooks to the autograd graph
    return result
```

The roles are split:

| | Who writes it | Who calls it |
|---|---|---|
| `forward` | You | `__call__` |
| `__call__` | `nn.Module` (inherited) | You, via `model(x)` |

#### Why not call `.forward()` directly

It returns the same result, but **skips the hooks**:

```python
y = model(x)           # correct: hooks fire
y = model.forward(x)   # works, but hooks are bypassed
```

Tools built on hooks (feature extraction, Captum, activation visualization, some profilers and quantization utilities) silently stop working if you bypass `__call__`.


Hook types:

- `register_forward_pre_hook`: runs before `forward`, receives inputs.
- `register_forward_hook`: runs after `forward`, receives inputs and output.
- `register_full_backward_hook`: runs during backprop, receives gradients.



In [48]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 2)

    def forward(self, x):
        return self.fc(x)

model = MyModel()
out = model(torch.rand(1, 10))
print(out)

tensor([[-0.0106,  0.4198]], grad_fn=<AddmmBackward0>)


# train() and eval() in NN

### What they are

`model.train()` and `model.eval()` set the `self.training` flag on a module and, recursively, on all of its children. That is all they do. The flag is then read inside `forward` by layers whose behavior depends on the mode.

```python
def train(self, mode=True):
    self.training = mode
    for module in self.children():
        module.train(mode)
    return self

def eval(self):
    return self.train(False)
```

Both return the module itself, so chaining works: `model.eval().to(device)`.

### What they do NOT do

- They do not compute or disable gradients (use `torch.no_grad()` or `torch.inference_mode()`).
- They do not update weights (that is `optimizer.step()`).
- They do not change `requires_grad`.
- They do not freeze anything. Training in `eval()` mode is still possible.

### Which layers are affected

| Layer | `train()` | `eval()` |
|---|---|---|
| `nn.Dropout`, `Dropout1d/2d/3d`, `AlphaDropout` | randomly zeroes activations | identity |
| `nn.BatchNorm1d/2d/3d`, `SyncBatchNorm` | uses batch statistics, updates `running_mean` / `running_var` | uses stored running statistics |
| `nn.RReLU` | random negative slope | fixed slope (mean) |
| `nn.LSTM`, `nn.GRU`, `nn.RNN` | inter-layer dropout active (if `dropout > 0`) | no dropout |
| `nn.MultiheadAttention`, `nn.TransformerEncoderLayer` | attention/layer dropout active | no dropout (fast path may be used) |
| `nn.InstanceNorm*` | only differs if `track_running_stats=True` | same |
| Custom modules (DropPath, DropBlock, noise layers) | depends on `if self.training` | depends on `if self.training` |

Not affected: `nn.Linear`, `nn.Conv*`, `nn.ReLU`, `nn.GELU`, `nn.Softmax`, `nn.Embedding`, `nn.LayerNorm`, `nn.GroupNorm`, pooling layers. `nn.Sequential` just propagates the flag to its children.

### How a module uses the flag

```python
class MyModule(nn.Module):
    def forward(self, x):
        if self.training:
            return x * 2    # train behavior
        return x            # eval behavior
```

Functional calls do **not** read the flag by themselves. Pass it explicitly:

```python
F.dropout(x, p=0.5, training=self.training)   # correct
F.dropout(x, p=0.5)                           # always active (training=True by default)
```

In [49]:
model.train()
model.eval()

MyModel(
  (fc): Linear(in_features=10, out_features=2, bias=True)
)

# apply() in NN

### What it is

`module.apply(fn)` calls the function `fn` on **every submodule** of the module (recursively, children first) and then on the module itself. It returns the module, so chaining works.

```python
def apply(self, fn):
    for module in self.children():
        module.apply(fn)
    fn(self)
    return self
```

`fn` receives one argument: a module. Whatever you want to do (initialize, freeze, print, count) goes inside `fn`.

----


### Order of traversal

Children are visited before the parent (post-order). For a model like this:

```python
net = nn.Sequential(nn.Linear(2, 3), nn.ReLU(), nn.Sequential(nn.Linear(3, 1)))
net.apply(lambda m: print(type(m).__name__))
```

the output is:

```
Linear
ReLU
Linear
Sequential
Sequential
```

The innermost modules come first and the outermost container comes last.

---

### Most common use: weight initialization

```python
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.ones_(m.weight)
        nn.init.zeros_(m.bias)

model.apply(init_weights)
```

Since `fn` is called on **every** module (including containers like `nn.Sequential`), always filter with `isinstance`, otherwise you may try to access `.weight` on a module that has none.


In [52]:
def init(module):
    if isinstance(module, nn.Linear):
        print(f'Model have linear')

model.apply(init)

Model have linear


MyModel(
  (fc): Linear(in_features=10, out_features=2, bias=True)
)